# UNSW-NB15 Innovation 1: CSG-HALT

**Calibration-guided Sparse Linear Attention Hardware Generation**  
**校准引导的稀疏 Linear Attention 硬件生成方法**

本实验借鉴 ViT4Mal 的边缘 Transformer 系统拆分和硬件约束驱动思想，但不照搬其模型。本实验对象是 UNSW-NB15 binary anomaly detection 的 Linear Transformer core；不改变输入输出 shape、网络层级或权重维度。

当前 notebook 仅执行 Stage 0-4。所有核心逻辑位于 `experiments/innovation1_csg_halt/scripts/`，本 notebook 只调用脚本并展示结果。Stage 5-8 在候选通过校准门槛后另行开展。


## 1. Path configuration

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path("/home/cym/prj2/finn/notebooks/icl_thesis-master")
EXP_ROOT = PROJECT_ROOT / "experiments" / "innovation1_csg_halt"
SCRIPT_DIR = EXP_ROOT / "scripts"
RESULTS_DIR = EXP_ROOT / "results"
LOG_DIR = EXP_ROOT / "logs" / "notebook_stage_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

assert EXP_ROOT.exists(), f"Missing experiment directory: {EXP_ROOT}"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("EXP_ROOT:", EXP_ROOT)


PROJECT_ROOT: /home/cym/prj2/finn/notebooks/icl_thesis-master
EXP_ROOT: /home/cym/prj2/finn/notebooks/icl_thesis-master/experiments/innovation1_csg_halt


## 2. Stage runner

In [2]:
def run_stage(script_name):
    script = SCRIPT_DIR / script_name
    assert script.exists(), f"Missing stage script: {script}"
    result = subprocess.run(
        [sys.executable, str(script)],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )
    log_path = LOG_DIR / f"{script.stem}.log"
    combined = f"STDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
    log_path.write_text(combined, encoding="utf-8")
    if result.returncode != 0:
        tail = "\n".join(combined.splitlines()[-80:])
        print(tail)
        raise RuntimeError(f"Stage failed ({result.returncode}): {script_name}; log={log_path}")
    print(result.stdout.strip())
    print("Log:", log_path)
    return result


def show_markdown(path):
    path = Path(path)
    assert path.exists(), f"Missing result: {path}"
    display(Markdown(path.read_text(encoding="utf-8")))


## 3. Stage 0: Snapshot baseline

In [3]:
run_stage("00_snapshot_baseline.py")
snapshot_path = RESULTS_DIR / "baseline_snapshot" / "baseline_file_hashes.json"
snapshot = json.loads(snapshot_path.read_text(encoding="utf-8"))
print("Protected files:", len(snapshot["files"]))
print("Integrity check: True (snapshot script exits nonzero on mismatch)")
pd.DataFrame(snapshot["files"].values()).loc[:, ["path", "sha256", "size_bytes"]]


Protected files: 13
Integrity OK: True
Snapshot: /home/cym/prj2/finn/notebooks/icl_thesis-master/experiments/innovation1_csg_halt/results/baseline_snapshot/baseline_file_hashes.json
Source-level baseline copy: /home/cym/prj2/finn/notebooks/icl_thesis-master/experiments/innovation1_csg_halt/copied_baseline/unsw_linear_transformer_hls_float_copy
Log: /home/cym/prj2/finn/notebooks/icl_thesis-master/experiments/innovation1_csg_halt/logs/notebook_stage_logs/00_snapshot_baseline.log
Protected files: 13
Integrity check: True (snapshot script exits nonzero on mismatch)


,path,sha256,size_bytes
0,/home/cym/prj2/finn/notebooks/icl_thesis-maste...,6599dd14c95e72522d84dcba82aae304f229e3421157f4...,18283
1,/home/cym/prj2/finn/notebooks/icl_thesis-maste...,7a8ba6fd6f9e851e0d6d63f93afc334b85f1c5eb307883...,196736
2,/home/cym/prj2/finn/notebooks/icl_thesis-maste...,40a1bfe1e62cd465d96752d85df09e5ba9dbcd6e562fe7...,2176
3,/home/cym/prj2/finn/notebooks/icl_thesis-maste...,8e18e6fc510a8ffb420caf21c1604ba27b3332931eac8f...,2227
4,/home/cym/prj2/finn/notebooks/icl_thesis-maste...,fd93ec2357d031a8d171709434f789708938157633e4d4...,136
5,/home/cym/prj2/finn/notebooks/icl_thesis-maste...,ff9363e88873d33d191680b0ff5753caf05336ee2e980a...,8795
6,/home/cym/prj2/finn/notebooks/icl_thesis-maste...,4f9d14ccc588564aea7c01a012c78d9b78ef62a3819f09...,316
7,/home/cym/prj2/finn/notebooks/icl_thesis-maste...,2d1bf93ecf5cb499d9af068bc295464ece999784402515...,53430
8,/home/cym/prj2/finn/notebooks/icl_thesis-maste...,7a9342ac14bb711ef2725a0e38d293b9debebc2e9837a1...,3068
9,/home/cym/prj2/finn/notebooks/icl_thesis-maste...,b77acb71f4a630d7b69c304ecf79e0f25d6f49c64ae3dd...,58505


## 4. Stage 1: Hardware bottleneck profiling

In [4]:
run_stage("01_profile_float_hls_bottleneck.py")
show_markdown(RESULTS_DIR / "bottleneck_profile" / "bottleneck_profile.md")


{
  "latency_max_cycles": 7869,
  "interval_max_cycles": 7870,
  "estimated_clock_period_ns": 8.567,
  "BRAM_18K": 176,
  "DSP48E": 258,
  "FF": 71828,
  "LUT": 65546,
  "resource_utilization": {
    "BRAM_18K": {
      "used": 176,
      "capacity": 280,
      "utilization_percent": 62.857142857142854
    },
    "DSP48E": {
      "used": 258,
      "capacity": 220,
      "utilization_percent": 117.27272727272727
    },
    "FF": {
      "used": 71828,
      "capacity": 106400,
      "utilization_percent": 67.50751879699249
    },
    "LUT": {
      "used": 65546,
      "capacity": 53200,
      "utilization_percent": 123.20676691729324
    }
  },
  "dsp_over_limit": true,
  "lut_over_limit": true,
  "operator_and_schedule_signals": {
    "exp_calls_in_source": 1,
    "sqrt_calls_in_source": 2,
    "division_expressions_in_source": 4,
    "vitis_warning_count": 37,
    "ii_violation_count": 36
  }
}
Profile: /home/cym/prj2/finn/notebooks/icl_thesis-master/experiments/innovation1_csg_hal

# Float HLS Baseline Bottleneck Profile

## Measured Baseline

| Metric | Value |
|---|---:|
| Latency max | 7869 cycles |
| Interval max | 7870 cycles |
| Estimated clock | 8.567 ns |
| BRAM_18K | 176 / 280 (62.86%) |
| DSP48E | 258 / 220 (117.27%) |
| FF | 71828 / 106400 (67.51%) |
| LUT | 65546 / 53200 (123.21%) |

The float baseline is functionally correct and meets the 10 ns target, but DSP and LUT
exceed PYNQ-Z2 capacity. It must not proceed directly to Vivado or board evaluation.

## Likely Hardware Bottlenecks

1. Dense Q/K/V projections replicate floating-point multiply/add operators.
2. The `ELU+1` feature map requires floating-point exponential hardware.
3. `K^T V` and the numerator perform dense Linear Attention accumulations.
4. The denominator combines dense Q/K channels and floating-point division.
5. The attention output projection and classifier retain dense floating-point paths.
6. Automatic loop unrolling and array partitioning replicate operators aggressively.
7. The log contains 36 II-violation messages and 37 warnings,
   showing memory-port/dependence pressure in addition to resource overuse.

## Innovation-1 Direction

The next step is calibration-guided sparse Linear Attention hardware generation, not
immediate Vivado integration. Dynamic event gates are used only to discover important
`phi(Q)`/`phi(K)` channels; the later HLS kernel will use static active-channel loop bounds.


## 5. Stage 2: PS/PL task partition plan

In [5]:
show_markdown(RESULTS_DIR / "bottleneck_profile" / "ps_pl_partition_plan.md")


# PS/PL Task Partition Plan

This design borrows the edge-deployment system partitioning idea from ViT4Mal without
copying its image model.

## Processing System (PS)

- Read UNSW-NB15 CSV files or `sample_inputs.npy`.
- Apply the saved OneHotEncoder, median imputation, StandardScaler, padding, and reshape.
- Schedule samples and transfer `[8, 24]` tensors to the accelerator.
- Receive two logits and apply argmax.
- Calculate binary anomaly-detection Accuracy, Precision, Recall, F1, and Confusion Matrix.
- Interpret `label=0` as normal and `label=1` as anomaly/attack.

## Programmable Logic (PL)

- Deploy only the Linear Transformer inference core.
- Fixed input shape: `[8, 24]`.
- Fixed output shape: `[2]` logits.
- No CSV parsing, sklearn preprocessing, sample scheduling, or metric calculation in PL.

## Relation to ViT4Mal

ViT4Mal partitions an edge image-malware Transformer system. CSG-HALT transfers that
system-level principle to UNSW-NB15 tabular network anomaly detection, while the optimized
hardware object is a softmax-free Linear Attention kernel rather than a ViT encoder.


## 6. Stage 3: Dynamic-threshold spike-gate calibration

In [6]:
run_stage("02_calibrate_dynamic_threshold_spikegate.py")
dynamic_path = RESULTS_DIR / "pytorch_calibration" / "dynamic_threshold_results.csv"
dynamic_df = pd.read_csv(dynamic_path)
dynamic_df.sort_values(
    ["passes_quality_gate", "prediction_match_rate_vs_original", "f1_delta_vs_original", "q_active_ratio", "k_active_ratio"],
    ascending=[False, False, False, True, True],
).loc[:, [
    "config_id", "method", "threshold_param", "f1", "f1_delta_vs_original",
    "prediction_match_rate_vs_original", "q_active_ratio", "k_active_ratio",
    "denominator_near_zero_count", "passes_quality_gate",
]].head(20)


Original metrics: {
  "accuracy": 0.609375,
  "precision": 0.11504424778761062,
  "recall": 1.0,
  "f1": 0.20634920634920634,
  "confusion_matrix": [
    [
      143,
      100
    ],
    [
      0,
      13
    ]
  ]
}
Passing configurations: 5
Best configuration: {
  "config_id": "dt_21_homeostatic_0p99",
  "method": "homeostatic",
  "threshold_param": 0.99,
  "accuracy": 0.6015625,
  "precision": 0.11304347826086956,
  "recall": 1.0,
  "f1": 0.203125,
  "confusion_matrix": [
    [
      141,
      102
    ],
    [
      0,
      13
    ]
  ],
  "f1_delta_vs_original": -0.0032242063492063378,
  "prediction_match_rate_vs_original": 0.9921875,
  "max_abs_logits_error_vs_original": 0.7883603572845459,
  "mean_abs_logits_error_vs_original": 0.03745082364184782,
  "q_active_ratio": 0.989471435546875,
  "k_active_ratio": 0.99005126953125,
  "estimated_attention_mac_reduction": 0.0102386474609375,
  "denominator_min": 83.77188873291016,
  "denominator_mean": 124.59359741210938,
  "denominat

,config_id,method,threshold_param,f1,f1_delta_vs_original,prediction_match_rate_vs_original,q_active_ratio,k_active_ratio,denominator_near_zero_count,passes_quality_gate
4,dt_04_percentile_1,percentile,1.00,0.206349,0.000000,1.000000,0.989990,0.989990,0,True
3,dt_03_percentile_0p5,percentile,0.50,0.206349,0.000000,1.000000,0.994965,0.994934,0,True
2,dt_02_percentile_0p25,percentile,0.25,0.206349,0.000000,1.000000,0.997498,0.997406,0,True
1,dt_01_percentile_0p1,percentile,0.10,0.206349,0.000000,1.000000,0.998932,0.998993,0,True
21,dt_21_homeostatic_0p99,homeostatic,0.99,0.203125,-0.003224,0.992188,0.989471,0.990051,0,True
0,dt_00_percentile_0,percentile,0.00,0.206349,0.000000,1.000000,0.999939,0.999969,0,False
5,dt_05_percentile_2,percentile,2.00,0.198473,-0.007876,0.980469,0.979980,0.979858,0,False
22,dt_22_homeostatic_0p98,homeostatic,0.98,0.198473,-0.007876,0.980469,0.980042,0.980286,0,False
6,dt_06_percentile_3,percentile,3.00,0.195489,-0.010860,0.972656,0.969910,0.969452,0,False
13,dt_13_dynamic_mad_m0p5,dynamic_mad,-0.50,0.196970,-0.009380,0.968750,0.697845,0.658722,0,False


In [7]:
show_markdown(RESULTS_DIR / "pytorch_calibration" / "dynamic_threshold_summary.md")


# Dynamic-Threshold Spike-Gate Calibration

- Samples: 256
- Task: UNSW-NB15 binary anomaly detection (`0=normal`, `1=anomaly/attack`)
- Original F1 on calibration samples: 0.206349
- No-gate wrapper maximum error: 0
- Tested configurations: 29
- Passing configurations: 5
- Best configuration: `dt_21_homeostatic_0p99`

| Config | F1 | F1 delta | Prediction match | Q active | K active | Pass |
|---|---:|---:|---:|---:|---:|---:|
| dt_21_homeostatic_0p99 | 0.203125 | -0.003224 | 0.992188 | 0.9895 | 0.9901 | True |
| dt_04_percentile_1 | 0.206349 | 0.000000 | 1.000000 | 0.9900 | 0.9900 | True |
| dt_03_percentile_0p5 | 0.206349 | 0.000000 | 1.000000 | 0.9950 | 0.9949 | True |
| dt_02_percentile_0p25 | 0.206349 | 0.000000 | 1.000000 | 0.9975 | 0.9974 | True |
| dt_01_percentile_0p1 | 0.206349 | 0.000000 | 1.000000 | 0.9989 | 0.9990 | True |
| dt_20_dynamic_mad_1p5 | 0.289855 | 0.083506 | 0.738281 | 0.1036 | 0.1049 | False |
| dt_19_dynamic_mad_1p25 | 0.310345 | 0.103996 | 0.734375 | 0.1431 | 0.1620 | False |
| dt_18_dynamic_mad_1p0 | 0.338983 | 0.132634 | 0.738281 | 0.1808 | 0.2220 | False |
| dt_17_dynamic_mad_0p75 | 0.301370 | 0.095021 | 0.792969 | 0.2467 | 0.2906 | False |
| dt_16_dynamic_mad_0p5 | 0.244444 | 0.038095 | 0.804688 | 0.3576 | 0.3615 | False |

The spike-inspired gate is a calibration instrument, not the final HLS implementation.
Only a passing configuration may be used to derive static active-channel sets. The gate
does not remove softmax because this Linear Attention model has no softmax.


## 7. Stage 4: Calibration-guided active-channel selection

In [8]:
thresholds = json.loads(
    (RESULTS_DIR / "pytorch_calibration" / "thresholds.json").read_text(encoding="utf-8")
)
if thresholds.get("best_config_id") is None:
    raise RuntimeError(
        "No dynamic configuration meets prediction match >= 0.99, F1 delta >= -0.01, "
        "and denominator_near_zero_count == 0. Stop before HLS."
    )
run_stage("03_select_active_channels.py")
active_df = pd.read_csv(RESULTS_DIR / "active_channel_selection" / "active_channel_eval.csv")
active_df.loc[:, [
    "candidate", "accuracy", "precision", "recall", "f1", "f1_delta_vs_original",
    "prediction_match_rate_vs_original", "q_active_ratio", "k_active_ratio",
    "denominator_near_zero_count", "passes_quality_gate",
]]


Source dynamic config: dt_21_homeostatic_0p99
Passing static sets: 3
Selected candidate: {
  "candidate": "active_87p5",
  "requested_active_ratio": 0.875,
  "accuracy": 0.609375,
  "precision": 0.11504424778761062,
  "recall": 1.0,
  "f1": 0.20634920634920634,
  "confusion_matrix": [
    [
      143,
      100
    ],
    [
      0,
      13
    ]
  ],
  "f1_delta_vs_original": 0.0,
  "prediction_match_rate_vs_original": 1.0,
  "max_abs_logits_error_vs_original": 0.4506722092628479,
  "mean_abs_logits_error_vs_original": 0.07437273737741634,
  "q_active_ratio": 0.875,
  "k_active_ratio": 0.875,
  "denominator_min": 97.03277587890625,
  "denominator_mean": 103.06040954589844,
  "denominator_near_zero_count": 0,
  "passes_quality_gate": true
}
Evaluation: /home/cym/prj2/finn/notebooks/icl_thesis-master/experiments/innovation1_csg_halt/results/active_channel_selection/active_channel_eval.csv
Log: /home/cym/prj2/finn/notebooks/icl_thesis-master/experiments/innovation1_csg_halt/logs/noteboo

,candidate,accuracy,precision,recall,f1,f1_delta_vs_original,prediction_match_rate_vs_original,q_active_ratio,k_active_ratio,denominator_near_zero_count,passes_quality_gate
0,active_100,0.609375,0.115044,1.000000,0.206349,0.000000,1.000000,1.0000,1.0000,0,True
1,active_93p75,0.609375,0.115044,1.000000,0.206349,0.000000,1.000000,0.9375,0.9375,0,True
2,active_87p5,0.609375,0.115044,1.000000,0.206349,0.000000,1.000000,0.8750,0.8750,0,True
3,active_81p25,0.621094,0.118182,1.000000,0.211382,0.005033,0.988281,0.8125,0.8125,0,False
4,active_75,0.636719,0.122642,1.000000,0.218487,0.012138,0.972656,0.7500,0.7500,0,False
5,active_62p5,0.664062,0.131313,1.000000,0.232143,0.025794,0.945312,0.6250,0.6250,0,False
6,active_50,0.679688,0.129032,0.923077,0.226415,0.020066,0.921875,0.5000,0.5000,0,False
7,active_37p5,0.640625,0.123810,1.000000,0.220339,0.013990,0.968750,0.3750,0.3750,0,False
8,active_25,0.597656,0.112069,1.000000,0.201550,-0.004799,0.988281,0.2500,0.2500,0,False


In [9]:
show_markdown(RESULTS_DIR / "active_channel_selection" / "active_channel_summary.md")
active_sets = json.loads(
    (RESULTS_DIR / "active_channel_selection" / "active_channel_sets.json").read_text(encoding="utf-8")
)
selected = active_sets.get("selected_candidate")
if selected is None:
    raise RuntimeError("No sparse static active-channel set passed. Stop before HLS generation.")
print("Selected sparse candidate:", json.dumps(selected, indent=2))


# Calibration-Guided Active-Channel Selection

- Dynamic calibration source: `dt_21_homeostatic_0p99`
- Evaluated samples: 256
- Passing static sets: 3
- Passing sparse static sets: 2
- Selected HLS candidate: `active_87p5`

| Candidate | Accuracy | Precision | Recall | F1 | F1 delta | Pred. match | Q active | K active | Pass |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| active_100 | 0.609375 | 0.115044 | 1.000000 | 0.206349 | 0.000000 | 1.000000 | 1.000 | 1.000 | True |
| active_93p75 | 0.609375 | 0.115044 | 1.000000 | 0.206349 | 0.000000 | 1.000000 | 0.938 | 0.938 | True |
| active_87p5 | 0.609375 | 0.115044 | 1.000000 | 0.206349 | 0.000000 | 1.000000 | 0.875 | 0.875 | True |
| active_81p25 | 0.621094 | 0.118182 | 1.000000 | 0.211382 | 0.005033 | 0.988281 | 0.812 | 0.812 | False |
| active_75 | 0.636719 | 0.122642 | 1.000000 | 0.218487 | 0.012138 | 0.972656 | 0.750 | 0.750 | False |
| active_62p5 | 0.664062 | 0.131313 | 1.000000 | 0.232143 | 0.025794 | 0.945312 | 0.625 | 0.625 | False |
| active_50 | 0.679688 | 0.129032 | 0.923077 | 0.226415 | 0.020066 | 0.921875 | 0.500 | 0.500 | False |
| active_37p5 | 0.640625 | 0.123810 | 1.000000 | 0.220339 | 0.013990 | 0.968750 | 0.375 | 0.375 | False |
| active_25 | 0.597656 | 0.112069 | 1.000000 | 0.201550 | -0.004799 | 0.988281 | 0.250 | 0.250 | False |

The selected Q/K indices are static hardware-generation metadata. A future sparse HLS
variant must reduce active loop bounds; multiplying a full 16-channel loop by a mask would
not count as sparse hardware generation.

Selected Q channels: `[0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15]`

Selected K channels: `[0, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15]`


Selected sparse candidate: {
  "candidate": "active_87p5",
  "requested_active_ratio": 0.875,
  "accuracy": 0.609375,
  "precision": 0.11504424778761062,
  "recall": 1.0,
  "f1": 0.20634920634920634,
  "confusion_matrix": [
    [
      143,
      100
    ],
    [
      0,
      13
    ]
  ],
  "f1_delta_vs_original": 0.0,
  "prediction_match_rate_vs_original": 1.0,
  "max_abs_logits_error_vs_original": 0.4506722092628479,
  "mean_abs_logits_error_vs_original": 0.07437273737741634,
  "q_active_ratio": 0.875,
  "k_active_ratio": 0.875,
  "denominator_min": 97.03277587890625,
  "denominator_mean": 103.06040954589844,
  "denominator_near_zero_count": 0,
  "passes_quality_gate": true
}


## 8. Stage 5-8 status

Deferred in this iteration, by design:

- Stage 5: Generate sparse mixed-fixed-point HLS variants
- Stage 6: HLS CSIM
- Stage 7: HLS CSYNTH
- Stage 8: Multi-objective candidate selection

No existing float HLS source, ONNX file, checkpoint, CSIM report, or CSYNTH report is modified by Stage 0-4.


## 9. Current conclusion

In [10]:
profile = json.loads(
    (RESULTS_DIR / "bottleneck_profile" / "bottleneck_profile.json").read_text(encoding="utf-8")
)
best_dynamic = json.loads(
    (RESULTS_DIR / "pytorch_calibration" / "dynamic_threshold_results.json").read_text(encoding="utf-8")
).get("best_config")
selected_set = active_sets["sets"][selected["candidate"]]

summary = f"""### Stage 0-4 summary

1. **Float baseline cannot enter PYNQ-Z2 implementation unchanged:** DSP {profile['DSP48E']}/220 and LUT {profile['LUT']}/53200 exceed the board budget.
2. **Dynamic spike-gate calibration is useful as an analysis tool:** `{best_dynamic['config_id']}` passes the strict quality gate with prediction match {best_dynamic['prediction_match_rate_vs_original']:.6f} and F1 delta {best_dynamic['f1_delta_vs_original']:.6f}.
3. **Static sparse candidate found:** `{selected['candidate']}` keeps {selected_set['active_count']}/16 Q channels and {selected_set['active_count']}/16 K channels; calibration prediction match is {selected['prediction_match_rate_vs_original']:.6f}, F1 delta is {selected['f1_delta_vs_original']:.6f}, and denominator near-zero count is {selected['denominator_near_zero_count']}.
4. **HLS status:** no sparse HLS variant has been generated or run yet. CSIM/CSYNTH and board feasibility are therefore not claimed.
5. **Next experiment:** generate mixed-fixed-point/feature-map/scheduling variants from the static Q/K indices in a copied HLS workspace, then admit only CSIM PASS variants to CSYNTH.

The 256 stored samples are used here for calibration and prediction-consistency checks. Their metrics are not presented as final PYNQ-Z2 board accuracy.
"""
display(Markdown(summary))


### Stage 0-4 summary

1. **Float baseline cannot enter PYNQ-Z2 implementation unchanged:** DSP 258/220 and LUT 65546/53200 exceed the board budget.
2. **Dynamic spike-gate calibration is useful as an analysis tool:** `dt_21_homeostatic_0p99` passes the strict quality gate with prediction match 0.992188 and F1 delta -0.003224.
3. **Static sparse candidate found:** `active_87p5` keeps 14/16 Q channels and 14/16 K channels; calibration prediction match is 1.000000, F1 delta is 0.000000, and denominator near-zero count is 0.
4. **HLS status:** no sparse HLS variant has been generated or run yet. CSIM/CSYNTH and board feasibility are therefore not claimed.
5. **Next experiment:** generate mixed-fixed-point/feature-map/scheduling variants from the static Q/K indices in a copied HLS workspace, then admit only CSIM PASS variants to CSYNTH.

The 256 stored samples are used here for calibration and prediction-consistency checks. Their metrics are not presented as final PYNQ-Z2 board accuracy.
